# CHB-MIT centre versus surround control

Self contained. Definitions below are copied verbatim from NeuroState_multiseed.ipynb
so this notebook runs on its own kernel.

The submitted paper reported elevated boundary activation in the centre epoch of
seizure windows in 53 of 53 cases. That statistic was computed only on seizure
windows, so it cannot separate a seizure effect from a property shared by every
window. The TUAB evaluation computed the same statistic for both classes and both
came out negative. This notebook computes both classes on CHB-MIT.

Requires the checkpoints written by the multi seed sweep at
models/multiseed/chb_mit_seed*.pt. Runtime is dominated by reading the HDF5 array;
inference over the 1,496 test windows takes about ten seconds per checkpoint.

In [1]:
# IMPORTS

import json
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import mannwhitneyu

print(torch.__version__, torch.cuda.is_available())

2.13.0+cu130 True


## Model definitions

In [2]:
# MODEL DEFINITIONS

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=22, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)


class ContrastiveBoundaryModule(nn.Module):
    """Boundaries from representation contrast, not absolute MLP."""

    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.n_scales = len(scales)
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
            for _ in scales])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_scales, self.n_scales * 2), nn.GELU(),
            nn.Linear(self.n_scales * 2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def _compute_contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = proj(x)
        h_norm = F.normalize(h, dim=-1)
        if offset < T:
            h_shifted = torch.roll(h_norm, -offset, dims=1)
            h_shifted[:, -offset:, :] = h_norm[:, -offset:, :]
            similarity = (h_norm * h_shifted).sum(dim=-1)
            contrast = 1.0 - (similarity + 1.0) / 2.0
            contrast = torch.sigmoid(
                (contrast - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            contrast = torch.zeros(B, T, device=x.device)
        return contrast

    def forward(self, x):
        per_scale = [self._compute_contrast(x, proj, off)
                     for proj, off in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach())
                          for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': 0.01 * consistency}


class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.n_cross = n_cross
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        assert embed_dim % self.n_heads == 0
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def _build_regime_mask(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        return torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))

    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        same = self._build_regime_mask(boundaries).unsqueeze(1)
        cross = 1.0 - same
        h_ie = self.n_intra
        h_ce = h_ie + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h_ie] = same.expand(B, self.n_intra, T, T)
        mask[:, h_ie:h_ce] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)
        attn_w = F.softmax(attn, dim=-1)
        attn_w = self.attn_drop(attn_w)
        out = (attn_w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))
        return (out, attn_w) if return_attention else out


class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(
            embed_dim, n_intra, n_inter, n_cross, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout))

    def forward(self, x, boundaries, return_attention=False):
        normed = self.norm1(x)
        if return_attention:
            a, w = self.attn(normed, boundaries, True)
            x = x + a
            x = x + self.mlp(self.norm2(x))
            return x, w
        x = x + self.attn(normed, boundaries)
        x = x + self.mlp(self.norm2(x))
        return x


class MultiResolutionEncoder(nn.Module):
    """Encodes a single epoch at 100/50/25 Hz, merges via interpolation."""

    def __init__(self, n_channels=3, n_samples=3000,
                 embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100hz = PatchEmbedding(
            n_channels, n_samples, embed_dim, dropout=dropout)
        self.enc_50hz = PatchEmbedding(
            n_channels, n_samples // 2, embed_dim, dropout=dropout)
        self.enc_25hz = PatchEmbedding(
            n_channels, n_samples // 4, embed_dim, dropout=dropout)
        self.merge = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100hz.seq_len
        self.seq_len_50 = self.enc_50hz.seq_len
        self.seq_len_25 = self.enc_25hz.seq_len

    def forward(self, x):
        x_50 = x[:, :, ::2]
        x_25 = x[:, :, ::4]
        emb_100 = self.enc_100hz(x)
        emb_50 = self.enc_50hz(x_50)
        emb_25 = self.enc_25hz(x_25)
        T1 = emb_100.shape[1]
        emb_50_up = F.interpolate(
            emb_50.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        emb_25_up = F.interpolate(
            emb_25.permute(0, 2, 1), size=T1,
            mode='linear', align_corners=False).permute(0, 2, 1)
        merged = torch.cat([emb_100, emb_50_up, emb_25_up], dim=-1)
        return self.merge(merged)


class MultiResContrastiveNeuroState(nn.Module):
    """MultiRes MultiEpoch NeuroState with contrastive boundaries."""

    def __init__(self, n_channels=3, n_samples=3000, n_classes=5,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 n_intra=4, n_inter=2, n_cross=2,
                 contrast_scales=(1, 4, 16), cp_hidden=64,
                 n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes
        self.n_context = n_context_epochs
        self.mr_encoder = MultiResolutionEncoder(
            n_channels, n_samples, embed_dim, dropout)
        tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = tokens_per_epoch * n_context_epochs
        self.pos_embed = nn.Parameter(
            torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(
            torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)
        self.changepoint_module = ContrastiveBoundaryModule(
            embed_dim=embed_dim, hidden_dim=cp_hidden,
            scales=contrast_scales, dropout=dropout)
        self.blocks = nn.ModuleList([
            NeuroStateBlock(embed_dim, n_intra, n_inter, n_cross,
                            dropout=dropout)
            for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, n_classes))
        self.tokens_per_epoch = tokens_per_epoch
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"ContrastiveMultiRes: {n_params/1e6:.2f}M params, "
              f"{total_tokens} tokens")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, return_boundaries=False):
        B, N, C, T = x.shape
        epoch_embs = []
        for i in range(N):
            emb = self.mr_encoder(x[:, i])
            emb = emb + self.epoch_embed[:, i]
            epoch_embs.append(emb)
        full_seq = torch.cat(epoch_embs, dim=1)
        full_seq = self.pos_drop(full_seq + self.pos_embed)
        cp_out = self.changepoint_module(full_seq)
        boundaries = cp_out['boundaries']
        boundary_loss = cp_out['boundary_loss']
        for block in self.blocks:
            full_seq = block(full_seq, boundaries)
        full_seq = self.norm(full_seq)
        start = self.tokens_per_epoch * (N // 2)
        end = start + self.tokens_per_epoch
        center_tokens = full_seq[:, start:end, :]
        pooled = center_tokens.mean(dim=1)
        logits = self.head(pooled)
        out = {'logits': logits, 'boundary_loss': boundary_loss}
        if return_boundaries:
            out['boundaries'] = boundaries
            out['per_scale'] = cp_out['per_scale']
        return out

## CHB-MIT data

In [3]:
# CHB-MIT DATA DEFINITIONS

def load_chbmit_data(h5_path):
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() for s in f['subject_ids'][:]])
        sfreq = f.attrs['sfreq']
    print(f"Loaded: {epochs.shape[0]} epochs, {epochs.shape[1]} ch, {sfreq} Hz")
    print(f"  Seizure: {np.sum(labels == 1)}, Normal: {np.sum(labels == 0)}")
    subjects = np.unique(subject_ids)
    subj_sz = {s: int(np.sum(labels[subject_ids == s] == 1)) for s in subjects}
    return epochs, labels, subject_ids, subj_sz


def round_robin_split(subj_sz, seed=42):
    """Subjects sorted by seizure count, then dealt round robin.

    Deterministic and independent of seed, so the split is fixed by
    construction and matches the split behind the reported CHB-MIT results.
    """
    sorted_s = sorted(subj_sz.keys(), key=lambda s: subj_sz[s], reverse=True)
    splits = {'train': [], 'val': [], 'test': []}
    pattern = ['train', 'train', 'val', 'test'] * 4
    for i, s in enumerate(sorted_s):
        splits[pattern[i % len(pattern)]].append(s)
    for name, subjs in splits.items():
        total = sum(subj_sz[s] for s in subjs)
        print(f"  {name}: {subjs} ({total} seizure epochs)")
    return splits


class CHBMITOnsetDataset(Dataset):
    def __init__(self, epochs, labels, subject_ids, subject_list, n_context=3):
        mask = np.isin(subject_ids, subject_list)
        self.epochs = epochs[mask]
        self.labels = labels[mask]
        self.subject_ids = subject_ids[mask]
        self.n_ctx = n_context
        self.windows = []
        for subj in subject_list:
            subj_idx = np.where(self.subject_ids == subj)[0]
            if len(subj_idx) < n_context:
                continue
            for start in range(len(subj_idx) - n_context + 1):
                group = subj_idx[start:start + n_context]
                if np.all(np.diff(group) == 1):
                    center = n_context // 2
                    self.windows.append({
                        'indices': group,
                        'label': int(self.labels[group[center]]),
                        'epoch_labels': [int(self.labels[g]) for g in group],
                        'subject': subj,
                    })
        win_labels = [w['label'] for w in self.windows]
        print(f"  {len(self.windows)} windows ({sum(win_labels)} seizure, "
              f"{len(win_labels) - sum(win_labels)} normal)")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        x = np.stack([self.epochs[i] for i in w['indices']], axis=0)
        return {
            'epoch': torch.tensor(x, dtype=torch.float32),
            'label': torch.tensor(w['label'], dtype=torch.long),
            'epoch_labels': torch.tensor(w['epoch_labels'], dtype=torch.long),
        }

In [4]:
# CONFIG

CHB_MIT_CONFIG = {
    'h5_path': 'data/processed/chbmit_combined_seizure_detection.h5',
    'ckpt_dir': 'models/multiseed',
    'n_channels': 17,
    'n_samples': 3000,
    'n_classes': 2,
    'batch_size': 32,
    'embed_dim': 128,
    'n_layers': 4,
    'dropout': 0.1,
    'n_intra': 4,
    'n_inter': 2,
    'n_cross': 2,
    'contrast_scales': (1, 4, 16),
    'cp_hidden': 64,
    'n_context_epochs': 3,
}

EXPECTED_TEST_SUBJECTS = ['chb05', 'chb20', 'chb19']
EXPECTED_TEST_WINDOWS = 1496
EXPECTED_TEST_SEIZURE = 53

## Load the test split

Only the test dataset is built. The training and validation datasets are skipped
because `CHBMITOnsetDataset` copies its slice of the epoch array, and the full
array is roughly 3.9 GB. The split itself is deterministic, so skipping them does
not change which subjects land in test.

In [5]:
# BUILD THE TEST SPLIT

epochs, labels, subject_ids, subj_sz = load_chbmit_data(CHB_MIT_CONFIG['h5_path'])
splits = round_robin_split(subj_sz)

test_ds = CHBMITOnsetDataset(epochs, labels, subject_ids, splits['test'],
                             n_context=CHB_MIT_CONFIG['n_context_epochs'])
test_ld = DataLoader(test_ds, batch_size=CHB_MIT_CONFIG['batch_size'],
                     shuffle=False, num_workers=0)

n_seizure = sum(w['label'] for w in test_ds.windows)
if splits['test'] != EXPECTED_TEST_SUBJECTS:
    print(f"WARNING: test subjects {splits['test']} differ from the reported "
          f"split {EXPECTED_TEST_SUBJECTS}")
if len(test_ds) != EXPECTED_TEST_WINDOWS or n_seizure != EXPECTED_TEST_SEIZURE:
    print(f"WARNING: got {len(test_ds)} windows / {n_seizure} seizure, expected "
          f"{EXPECTED_TEST_WINDOWS} / {EXPECTED_TEST_SEIZURE}")
else:
    print("Test split matches the reported CHB-MIT evaluation")

Loaded: 9639 epochs, 17 ch, 100 Hz
  Seizure: 312, Normal: 9327
  train: ['chb15', 'chb12', 'chb01', 'chb03', 'chb14', 'chb17'] (195 seizure epochs)
  val: ['chb08', 'chb10', 'chb22'] (63 seizure epochs)
  test: ['chb05', 'chb20', 'chb19'] (54 seizure epochs)
  1496 windows (53 seizure, 1443 normal)
Test split matches the reported CHB-MIT evaluation


## Centre versus surround

In [6]:
# CENTRE VERSUS SURROUND

def build_model(cfg, device='cuda'):
    return MultiResContrastiveNeuroState(
        n_channels=cfg['n_channels'],
        n_samples=cfg['n_samples'],
        n_classes=cfg['n_classes'],
        embed_dim=cfg['embed_dim'],
        n_layers=cfg['n_layers'],
        dropout=cfg['dropout'],
        n_intra=cfg['n_intra'],
        n_inter=cfg['n_inter'],
        n_cross=cfg['n_cross'],
        contrast_scales=cfg['contrast_scales'],
        cp_hidden=cfg['cp_hidden'],
        n_context_epochs=cfg['n_context_epochs']).to(device)


def centre_surround_delta(model, loader, device='cuda'):
    """Mean boundary activation in the centre epoch minus the flanking epochs.

    Returns one delta per window together with the window label, so the
    statistic can be reported separately for seizure and normal windows.
    """
    tpe = model.tokens_per_epoch
    c0 = tpe * (model.n_context // 2)
    c1 = c0 + tpe
    model.eval()
    model.to(device)
    deltas, win_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch['epoch'].to(device), return_boundaries=True)
            b = out['boundaries'].cpu().numpy()
            centre = b[:, c0:c1].mean(axis=1)
            surround = np.concatenate([b[:, :c0], b[:, c1:]], axis=1).mean(axis=1)
            deltas.append(centre - surround)
            win_labels.append(batch['label'].numpy())
    return np.concatenate(deltas), np.concatenate(win_labels)


def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1))
                 / (na + nb - 2))
    return float((a.mean() - b.mean()) / sp) if sp > 0 else 0.0

In [7]:
# RUN

ckpt_dir = Path(CHB_MIT_CONFIG['ckpt_dir'])
ckpts = sorted(ckpt_dir.glob('chb_mit_seed*.pt'))

if not ckpts:
    print(f"No CHB-MIT checkpoints found in {ckpt_dir}")
    print("Expected files named chb_mit_seed<N>.pt written by the multi seed sweep.")
    per_seed = []
else:
    print(f"Found {len(ckpts)} checkpoints: {[c.name for c in ckpts]}")
    per_seed = []
    for ck in ckpts:
        model = build_model(CHB_MIT_CONFIG)
        model.load_state_dict(torch.load(ck, map_location='cpu'))

        d, y = centre_surround_delta(model, test_ld)
        ds, dn = d[y == 1], d[y == 0]
        u, p = mannwhitneyu(ds, dn, alternative='two-sided')

        row = {
            'checkpoint': ck.name,
            'n_seizure': int(len(ds)),
            'n_normal': int(len(dn)),
            'delta_seizure': float(ds.mean()),
            'delta_normal': float(dn.mean()),
            'positive_seizure': float((ds > 0).mean()),
            'positive_normal': float((dn > 0).mean()),
            'mannwhitney_u': float(u),
            'p_value': float(p),
            'cohens_d': cohens_d(ds, dn),
        }
        per_seed.append(row)
        print(f"{ck.name}: seizure delta {row['delta_seizure']:+.5f} "
              f"positive {row['positive_seizure']:.1%} | "
              f"normal delta {row['delta_normal']:+.5f} "
              f"positive {row['positive_normal']:.1%} | "
              f"p={p:.4f} d={row['cohens_d']:+.3f}")

    out_path = ckpt_dir / 'chb_mit_centre_surround.json'
    out_path.write_text(json.dumps(per_seed, indent=2))
    print(f"Saved to {out_path}")

Found 5 checkpoints: ['chb_mit_seed42.pt', 'chb_mit_seed43.pt', 'chb_mit_seed44.pt', 'chb_mit_seed45.pt', 'chb_mit_seed46.pt']
ContrastiveMultiRes: 1.07M params, 588 tokens
chb_mit_seed42.pt: seizure delta +0.00107 positive 100.0% | normal delta +0.00114 positive 100.0% | p=0.0001 d=-0.482
ContrastiveMultiRes: 1.07M params, 588 tokens
chb_mit_seed43.pt: seizure delta +0.00329 positive 100.0% | normal delta +0.00563 positive 100.0% | p=0.0000 d=-0.961
ContrastiveMultiRes: 1.07M params, 588 tokens
chb_mit_seed44.pt: seizure delta +0.00564 positive 98.1% | normal delta +0.00762 positive 99.9% | p=0.0000 d=-0.642
ContrastiveMultiRes: 1.07M params, 588 tokens
chb_mit_seed45.pt: seizure delta +0.01441 positive 96.2% | normal delta +0.02095 positive 99.6% | p=0.0000 d=-0.869
ContrastiveMultiRes: 1.07M params, 588 tokens
chb_mit_seed46.pt: seizure delta +0.00138 positive 100.0% | normal delta +0.00153 positive 100.0% | p=0.0000 d=-1.163
Saved to models/multiseed/chb_mit_centre_surround.json


## Verdict

The paper's original claim was that centre-epoch activation exceeds the
surrounding epochs in seizure windows. That claim only means something if normal
windows behave differently.

In [8]:
# VERDICT

if per_seed:
    pos_s = np.array([r['positive_seizure'] for r in per_seed])
    pos_n = np.array([r['positive_normal'] for r in per_seed])
    pvals = np.array([r['p_value'] for r in per_seed])
    ds_ = np.array([r['cohens_d'] for r in per_seed])

    print(f"seizure windows positive: {pos_s.mean():.1%} +/- {pos_s.std(ddof=1):.1%}")
    print(f"normal windows positive:  {pos_n.mean():.1%} +/- {pos_n.std(ddof=1):.1%}")
    print(f"Cohen's d: {ds_.mean():+.3f} +/- {ds_.std(ddof=1):.3f}")
    print(f"Mann-Whitney p: min {pvals.min():.4f}, max {pvals.max():.4f}")

    # Order matters. A significant difference is only evidence FOR the claim
    # if normal windows behave differently and the sign favours seizures.
    if pos_n.mean() > 0.8:
        print("\nNormal windows are also predominantly positive, so the "
              "original 53/53 figure describes every window rather than "
              "seizure windows. It reflects window geometry, not seizure "
              "content, and stays out of the paper.")
        if (pvals < 0.05).all() and ds_.mean() < 0:
            print("The difference that is significant runs the wrong way: "
                  "normal windows show the larger centre minus surround "
                  "delta in every seed.")
    elif (pvals < 0.05).all() and ds_.mean() > 0.3:
        print("\nSeizure windows separate from normal windows in the "
              "expected direction across all seeds. The effect is specific "
              "to seizure windows and can be reported with this effect size.")
    else:
        print("\nNo consistent separation favouring seizure windows. "
              "The claim stays out of the paper.")

seizure windows positive: 98.9% +/- 1.7%
normal windows positive:  99.9% +/- 0.2%
Cohen's d: -0.823 +/- 0.267
Mann-Whitney p: min 0.0000, max 0.0001

Normal windows are also predominantly positive, so the original 53/53 figure describes every window rather than seizure windows. It reflects window geometry, not seizure content, and stays out of the paper.
The difference that is significant runs the wrong way: normal windows show the larger centre minus surround delta in every seed.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>